# Repo Context Copilot: adaptive hybrid-RAG experiment
Pipeline: clone repo → AST chunking → bge-base embeddings (Astra DB) + BM25 → RRF hybrid search
→ cross-encoder rerank → adaptive cutoff → LLM answer.

Run top to bottom. Set `REINGEST = True` only when the chunker or embedding model changes.

In [14]:
import ast
import csv
import hashlib
import json
import logging
import os
import re
import shutil
import subprocess
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError
from dataclasses import dataclass, asdict, is_dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import bm25s
import tiktoken
from astrapy import DataAPIClient
from astrapy.constants import VectorMetric
from astrapy.info import CollectionDefinition
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from sentence_transformers import CrossEncoder, SentenceTransformer

load_dotenv()
logger = logging.getLogger(__name__)

## 0. Config

In [15]:
REPO_URL = "https://github.com/shreeragkh/Hybrid-Search-RAG"

# Embeddings: bge-base @256 ingests ~3x faster than bge-large with equal post-rerank quality in our eval.
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
EMBED_DIM = 768
EMBED_MAX_SEQ = 256
# One name used for BOTH ingestion and retrieval. Astra free tier allows ~100 indexes in total
# and every collection uses several, so drop stale collections you no longer need.
REPO_NAME = Path(urlparse(REPO_URL).path).stem
COLLECTION_NAME = re.sub(r"[^0-9a-zA-Z]+", "_", REPO_NAME).strip("_").lower()[:48]
REINGEST = False  # True: drop and rebuild the collection (required after any chunker/model change)

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
MAX_CHUNK_CHARS = 1500

# Hybrid search / RRF fusion
BM25_WEIGHT, VECTOR_WEIGHT, RRF_K = 0.4, 0.6, 60

# Pipeline
RELEVANCE_THRESHOLD = -8.5  # tuned to MiniLM cross-encoder raw logits; recalibrate if the reranker changes
MODEL_CONTEXT_WINDOW = 12000
ABSTENTION = "Cannot be determined from the provided repository context."
RETRIEVAL_BUDGET = {"LOW": 10, "MEDIUM": 20, "HIGH": 30}
DROPOFF_BY_COMPLEXITY = {"LOW": 0.08, "MEDIUM": 0.10, "HIGH": 0.12}
MIN_KEEP_BY_COMPLEXITY = {"LOW": 3, "MEDIUM": 4, "HIGH": 5}


def _openai(model: str, max_tokens: int, **kwargs) -> ChatOpenAI:
    return ChatOpenAI(model=model, temperature=0, max_tokens=max_tokens,
                      api_key=os.getenv("OPENAI_API_KEY"), **kwargs)


classifier_llm = _openai("gpt-5-nano", 128, reasoning_effort="minimal")
llm = _openai("gpt-5-mini", 256, reasoning_effort="minimal")
judge_llm = _openai("gpt-4.1-mini", 16)

## 1. Clone and chunk

In [16]:
def clone_repo(url: str, dest_root: str = "./temp/repos/") -> tuple[Path, str, str]:
    """Shallow-clone a GitHub repo (re-cloning cleanly if it exists). Returns (path, name, short_sha)."""
    repo_name = Path(urlparse(url).path).stem
    dest = Path(dest_root) / repo_name
    if dest.exists():
        shutil.rmtree(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(["git", "clone", "--depth", "1", url, str(dest)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed: {result.stderr.strip()}")

    sha = subprocess.run(["git", "-C", str(dest), "rev-parse", "HEAD"],
                         capture_output=True, text=True).stdout.strip()[:12]
    return dest, repo_name, sha


repo_path, repo_name, commit_sha = clone_repo(REPO_URL)
print(f"cloned {repo_name} ({commit_sha}) -> {repo_path}")

cloned Hybrid-Search-RAG (48cbc7a6fdac) -> temp/repos/Hybrid-Search-RAG


In [17]:
CODE_ONLY_MAP = {
    ".py": "python", ".js": "javascript", ".jsx": "javascript",
    ".ts": "typescript", ".tsx": "typescript", ".java": "java",
    ".go": "go", ".rb": "ruby", ".rs": "rust", ".c": "c", ".h": "c",
    ".cpp": "cpp", ".hpp": "cpp", ".cs": "csharp", ".php": "php",
}
DOC_EXTENSIONS = {".md": "markdown", ".rst": "restructuredtext", ".txt": "text"}
PRIORITY_DOC_FILENAMES = {"readme.md", "readme.rst", "readme.txt", "readme"}
EXCLUDE_DIRS = {".git", "node_modules", "venv", ".venv", "__pycache__", "dist", "build",
                ".next", "target", "vendor", ".idea", ".mypy_cache"}
EXCLUDE_FILENAMES = {"package-lock.json", "yarn.lock", "poetry.lock"}
EXCLUDE_PATTERNS = re.compile(r"\.min\.(js|css)$|\.d\.ts$|_pb2\.py$")

# Regex symbol splitters for non-Python languages (Python uses the AST instead).
GENERIC_FUNC_PATTERNS = {
    "javascript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "typescript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "java": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "go": re.compile(r"^\s*func\s+(\(\w+\s+\*?\w+\)\s+)?(\w+)\s*\("),
    "ruby": re.compile(r"^\s*def\s+(\w+)|^\s*class\s+(\w+)"),
    "rust": re.compile(r"^\s*(pub\s+)?fn\s+(\w+)|^\s*(pub\s+)?struct\s+(\w+)"),
    "c": re.compile(r"^\s*[\w\*]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "cpp": re.compile(r"^\s*[\w\*:<>]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "csharp": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "php": re.compile(r"^\s*function\s+(\w+)|^\s*class\s+(\w+)"),
}


@dataclass
class Chunk:
    repo: str
    file_path: str
    language: str
    symbol_type: str  # function | class | method | constant | block | doc_section
    symbol_name: str
    start_line: int
    end_line: int
    content: str
    char_count: int
    chunk_id: str = ""

    def __post_init__(self):
        if not self.chunk_id:
            raw = f"{self.repo}:{self.file_path}:{self.symbol_name}:{self.start_line}-{self.end_line}"
            self.chunk_id = hashlib.sha1(raw.encode()).hexdigest()[:16]


def discover_files(repo_dir: Path, max_file_kb: int = 500) -> list[Path]:
    files = []
    for root, dirs, filenames in os.walk(repo_dir):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS and not d.startswith(".")]
        for fn in filenames:
            if fn in EXCLUDE_FILENAMES or EXCLUDE_PATTERNS.search(fn):
                continue
            ext = Path(fn).suffix.lower()
            if not (ext in CODE_ONLY_MAP or ext in DOC_EXTENSIONS or fn.lower() in PRIORITY_DOC_FILENAMES):
                continue
            full = Path(root) / fn
            try:
                if full.stat().st_size > max_file_kb * 1024:
                    continue
            except OSError:
                continue
            files.append(full)
    return files


def split_oversized(chunk: Chunk, max_chars: int = MAX_CHUNK_CHARS) -> list[Chunk]:
    """Split a chunk into `<name>_partN` pieces of roughly max_chars each."""
    if chunk.char_count <= max_chars:
        return [chunk]
    out, buf, buf_start, cur_len = [], [], chunk.start_line, 0

    def emit():
        src = "\n".join(buf)
        out.append(Chunk(chunk.repo, chunk.file_path, chunk.language, chunk.symbol_type,
                         f"{chunk.symbol_name}_part{len(out) + 1}", buf_start,
                         buf_start + len(buf) - 1, src, len(src)))

    for i, line in enumerate(chunk.content.splitlines()):
        buf.append(line)
        cur_len += len(line) + 1
        if cur_len >= max_chars:
            emit()
            buf, buf_start, cur_len = [], chunk.start_line + i + 1, 0
    if buf:
        emit()
    return out


def chunk_generic_lines(path: Path, repo_name: str, text: str, language: str,
                        window: int = 60, overlap: int = 10) -> list[Chunk]:
    """Sliding-window fallback for files without symbol-level parsing."""
    lines = text.splitlines()
    chunks, i, n = [], 0, len(lines)
    while i < n:
        end = min(i + window, n)
        src = "\n".join(lines[i:end])
        if src.strip():
            chunks.append(Chunk(repo_name, str(path), language, "block",
                                f"lines_{i + 1}-{end}", i + 1, end, src, len(src)))
        if end == n:
            break
        i += window - overlap
    return chunks


def chunk_markdown_file(path: Path, repo_name: str) -> list[Chunk]:
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    header = re.compile(r"^#{1,3}\s+(.+)")
    starts = [i for i, line in enumerate(lines) if header.match(line)]
    if not starts:
        return chunk_generic_lines(path, repo_name, text, "markdown", window=80, overlap=10)

    chunks = []
    for idx, start in enumerate(starts):
        end = starts[idx + 1] - 1 if idx + 1 < len(starts) else len(lines) - 1
        while end > start and not lines[end].strip():
            end -= 1
        src = "\n".join(lines[start:end + 1])
        name = header.match(lines[start]).group(1).strip()
        chunks.append(Chunk(repo_name, str(path), "markdown", "doc_section",
                            name, start + 1, end + 1, src, len(src)))
    return chunks


def _node_start(node) -> int:
    """First line of a node including its decorators (ast gives the `def` line otherwise)."""
    return min([node.lineno] + [d.lineno for d in getattr(node, "decorator_list", [])])


def chunk_python_file(path: Path, repo_name: str) -> list[Chunk]:
    """One chunk per function / method / constant. Classes with methods get a header-only chunk
    (docstring + attributes) plus one chunk per method, so code is never indexed twice.
    Remaining top-level code becomes contiguous `module_level_<start>-<end>` blocks."""
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    try:
        tree = ast.parse(text)
    except SyntaxError:
        return chunk_generic_lines(path, repo_name, text, "python")

    chunks, covered = [], set()

    def add(kind, name, start, end):
        covered.update(range(start, end + 1))
        src = "\n".join(lines[start - 1:end])
        chunks.append(Chunk(repo_name, str(path), "python", kind, name, start, end, src, len(src)))

    funcs = (ast.FunctionDef, ast.AsyncFunctionDef)
    for node in tree.body:
        start, end = _node_start(node), node.end_lineno
        if isinstance(node, funcs):
            add("function", node.name, start, end)
        elif isinstance(node, ast.ClassDef):
            methods = [s for s in node.body if isinstance(s, funcs)]
            if methods:
                header_end = _node_start(methods[0]) - 1
                if header_end >= start:
                    add("class", node.name, start, header_end)
                for sub in methods:
                    add("method", f"{node.name}.{sub.name}", _node_start(sub), sub.end_lineno)
            else:
                add("class", node.name, start, end)
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            target = node.targets[0] if isinstance(node, ast.Assign) else node.target
            add("constant", getattr(target, "id", "constant"), start, end)

    run = []

    def flush():
        if run:
            body = lines[run[0] - 1:run[-1]]
            if any(l.strip() and not l.strip().startswith("#") for l in body):  # skip blank/comment-only runs
                src = "\n".join(body)
                chunks.append(Chunk(repo_name, str(path), "python", "block",
                                    f"module_level_{run[0]}-{run[-1]}", run[0], run[-1], src, len(src)))
            run.clear()

    for ln in range(1, len(lines) + 1):
        if ln in covered:
            flush()
        else:
            run.append(ln)
    flush()
    return chunks


def chunk_generic_symbols(path: Path, repo_name: str, language: str) -> list[Chunk]:
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    pattern = GENERIC_FUNC_PATTERNS[language]
    starts = [i for i, line in enumerate(lines) if pattern.search(line)]
    if not starts:
        return chunk_generic_lines(path, repo_name, text, language)

    chunks = []
    for idx, start in enumerate(starts):
        end = starts[idx + 1] - 1 if idx + 1 < len(starts) else len(lines) - 1
        while end > start and not lines[end].strip():
            end -= 1
        src = "\n".join(lines[start:end + 1])
        m = pattern.search(lines[start])
        name = next((g for g in m.groups() if g and re.match(r"^\w+$", g)), "anonymous")
        chunks.append(Chunk(repo_name, str(path), language, "function",
                            name, start + 1, end + 1, src, len(src)))
    return chunks


def chunk_file(path: Path, repo_name: str) -> list[Chunk]:
    ext = path.suffix.lower()
    language = CODE_ONLY_MAP.get(ext, "text")
    if language == "python":
        return chunk_python_file(path, repo_name)
    if language in GENERIC_FUNC_PATTERNS:
        return chunk_generic_symbols(path, repo_name, language)
    if ext == ".md" or path.name.lower().startswith("readme"):
        return chunk_markdown_file(path, repo_name)
    text = path.read_text(encoding="utf-8", errors="ignore")
    return chunk_generic_lines(path, repo_name, text, DOC_EXTENSIONS.get(ext, language))


def chunk_repo(repo_dir: Path, repo_name: str) -> list[Chunk]:
    return [piece for f in discover_files(repo_dir)
            for c in chunk_file(f, repo_name)
            for piece in split_oversized(c)]


chunks = chunk_repo(repo_path, repo_name)
print(f"{len(chunks)} chunks | max size {max(c.char_count for c in chunks)} chars | "
      f"avg {sum(c.char_count for c in chunks) / len(chunks):.0f}")
print("by type:", dict(Counter(c.symbol_type for c in chunks)))

184 chunks | max size 1620 chars | avg 500
by type: {'doc_section': 22, 'class': 17, 'constant': 28, 'function': 38, 'block': 39, 'method': 40}


## 2. Embeddings and Astra DB (vector store)

In [18]:
model = SentenceTransformer(EMBED_MODEL)
model.max_seq_length = EMBED_MAX_SEQ
assert model.get_sentence_embedding_dimension() == EMBED_DIM, "EMBED_DIM does not match EMBED_MODEL"

db = DataAPIClient().get_database(api_endpoint=os.getenv("API_ENDPOINT"), token=os.getenv("API_TOKEN"))


def ingest_chunks(chunks: list[Chunk], reingest: bool = False):
    """Embed locally and load into Astra. Reuses the collection if it already matches the chunks."""
    exists = COLLECTION_NAME in db.list_collection_names()
    if exists and reingest:
        db.drop_collection(COLLECTION_NAME)
        exists = False

    if exists:
        collection = db.get_collection(COLLECTION_NAME)
        n = collection.count_documents({}, upper_bound=100_000)
        if n != len(chunks):
            raise RuntimeError(f"'{COLLECTION_NAME}' holds {n} docs but there are {len(chunks)} chunks. "
                               "Set REINGEST = True (chunker or repo changed).")
        print(f"reusing '{COLLECTION_NAME}' ({n} docs)")
        return collection

    definition = (CollectionDefinition.builder()
                  .with_vector_dimension(EMBED_DIM)
                  .with_vector_metric(VectorMetric.COSINE)
                  .build())
    collection = db.create_collection(COLLECTION_NAME, definition=definition)

    t0 = time.monotonic()
    vectors = model.encode([c.content for c in chunks], normalize_embeddings=True,
                           show_progress_bar=True, batch_size=32).tolist()
    print(f"encode: {time.monotonic() - t0:.1f}s")

    docs = [{"_id": c.chunk_id, "$vector": v, **asdict(c)} for v, c in zip(vectors, chunks)]
    for i in range(0, len(docs), 50):
        collection.insert_many(docs[i:i + 50], request_timeout_ms=30000)
    print(f"inserted {len(docs)} chunks into '{COLLECTION_NAME}'")
    return collection


collection = ingest_chunks(chunks, REINGEST)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 435.10it/s]
/tmp/ipykernel_45760/3496836952.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == EMBED_DIM, "EMBED_DIM does not match EMBED_MODEL"
Batches: 100%|██████████| 6/6 [00:42<00:00,  7.14s/it]


encode: 43.0s
inserted 184 chunks into 'hybrid_search_rag'


## 3. BM25 (sparse index, keyed on the same chunk_id as the vectors)

In [19]:
class BM25Retriever:
    """bm25s index over chunk dicts/dataclasses; persisted per repo; results carry chunk_id for RRF."""

    def __init__(self, index_dir: str | Path = "bm25_index"):
        self.index_dir = Path(index_dir)
        self.retriever: bm25s.BM25 | None = None
        self.corpus: list[str] = []
        self.metadata: list[dict[str, Any]] = []

    def build(self, chunks: list[Any], text_key: str = "content") -> None:
        dicts = [asdict(c) if is_dataclass(c) else c for c in chunks]
        if not dicts:
            raise ValueError("Cannot build a BM25 index from an empty chunk list.")
        self.corpus = [d[text_key] for d in dicts]
        self.metadata = [{k: v for k, v in d.items() if k != text_key} for d in dicts]
        self.retriever = bm25s.BM25()
        self.retriever.index(bm25s.tokenize(self.corpus, show_progress=False), show_progress=False)

    def save(self) -> None:
        if self.retriever is None:
            raise RuntimeError("No index to save. Call build() first.")
        self.index_dir.mkdir(parents=True, exist_ok=True)
        self.retriever.save(str(self.index_dir), corpus=self.corpus)
        with open(self.index_dir / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(self.metadata, f)

    def load(self) -> None:
        if not self.index_dir.exists():
            raise FileNotFoundError(f"No index found at {self.index_dir}")
        self.retriever = bm25s.BM25.load(str(self.index_dir), load_corpus=True)
        with open(self.index_dir / "metadata.json", "r", encoding="utf-8") as f:
            self.metadata = json.load(f)
        self.corpus = [doc["text"] if isinstance(doc, dict) else doc for doc in self.retriever.corpus]
        self.retriever.corpus = None  # make retrieve() return plain indices

    def query(self, query_text: str, k: int = 10) -> list[dict[str, Any]]:
        if self.retriever is None:
            raise RuntimeError("Index not built or loaded. Call build() or load() first.")
        k = min(k, len(self.corpus))
        if k == 0:
            return []
        tokens = bm25s.tokenize(query_text, show_progress=False)
        indices, scores = self.retriever.retrieve(tokens, k=k, show_progress=False)
        results = []
        for idx, score in zip(indices[0], scores[0]):
            meta = self.metadata[int(idx)]
            results.append({"text": self.corpus[int(idx)], "score": float(score),
                            "chunk_id": meta.get("chunk_id"), "metadata": meta})
        return results


bm25 = BM25Retriever(index_dir=f"./bm25_index/{repo_name}")
bm25.build(chunks)
bm25.save()

## 4. Retrieval: vector, hybrid (RRF), reranker

In [20]:
class VectorRetriever:
    """Dense retrieval from Astra. `model` must be the same one used at ingestion time."""

    def __init__(self, collection, model):
        self.collection = collection
        self.model = model

    def query(self, query_text: str, k: int = 5) -> list[dict[str, Any]]:
        return self.retrieve(query_text, top_k=k)

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> list[dict[str, Any]]:
        """Errors propagate on purpose: HybridSearch degrades to BM25 and logs, and `preflight`
        turns a silently broken vector side into a hard failure before any eval."""
        vector = self.model.encode([query], normalize_embeddings=True).tolist()[0]
        results = self.collection.find(sort={"$vector": vector}, limit=top_k, include_similarity=True)

        docs = []
        for i, doc in enumerate(results, start=1):
            similarity = doc.get("$similarity", 0.0)
            if similarity < score_threshold:
                continue
            content = doc.get("content", "")
            docs.append({
                "id": doc.get("_id"),
                "chunk_id": doc.get("chunk_id", doc.get("_id")),
                "text": content,
                "content": content,
                "metadata": {k: v for k, v in doc.items()
                             if k not in ("_id", "$vector", "$similarity", "content")},
                "score": similarity,
                "similarity_score": similarity,
                "rank": i,
            })
        return docs


def reciprocal_rank_fusion(bm25_results, vector_results, bm25_weight, vector_weight, rrf_k):
    """Weighted RRF over two ranked lists, joined on chunk_id. Returns docs sorted by fused_score."""
    fused = {}
    for source, results, weight in (("bm25", bm25_results, bm25_weight),
                                    ("vector", vector_results, vector_weight)):
        for rank, doc in enumerate(results, start=1):
            text = doc.get("text") or doc.get("content") or ""
            doc_id = str(doc.get("chunk_id") or text.strip())
            entry = fused.setdefault(doc_id, {"chunk_id": doc_id, "text": text, "metadata": {},
                                              "bm25_score": 0.0, "vector_score": 0.0, "fused_score": 0.0})
            if not entry["metadata"]:
                entry["metadata"] = doc.get("metadata") or {}
            entry["fused_score"] += weight / (rrf_k + rank)
            entry[f"{source}_score"] = doc.get("similarity_score") or doc.get("score") or 0.0
    return sorted(fused.values(), key=lambda d: d["fused_score"], reverse=True)


class HybridSearchError(Exception):
    """Raised only when BOTH retrievers fail."""


class HybridSearch:
    """BM25 + vector retrieval run in parallel, fused with RRF. One retriever failing degrades gracefully."""

    def __init__(self, bm25_retriever, vector_retriever):
        self.bm25_retriever = bm25_retriever
        self.vector_retriever = vector_retriever

    def hybrid_retrieval(self, query_text: str, k: int = 10, fetch_k: int = 25,
                         bm25_weight: float = BM25_WEIGHT, vector_weight: float = VECTOR_WEIGHT,
                         rrf_k: int = RRF_K, timeout_s: float = 15.0) -> list[dict[str, Any]]:
        bm25_results, vector_results = self._run_retrievers(query_text, fetch_k, timeout_s)
        fused = reciprocal_rank_fusion(bm25_results, vector_results, bm25_weight, vector_weight, rrf_k)
        return fused[:k]

    def _run_retrievers(self, query_text: str, fetch_k: int, timeout_s: float):
        def safe_call(fn, name):
            try:
                return fn(query_text, k=fetch_k)
            except Exception:
                logger.exception("Retriever %s failed", name)
                return []

        with ThreadPoolExecutor(max_workers=2) as pool:
            futures = {"bm25": pool.submit(safe_call, self.bm25_retriever.query, "bm25"),
                       "vector": pool.submit(safe_call, self.vector_retriever.query, "vector")}
            results = {}
            for name, future in futures.items():
                try:
                    results[name] = future.result(timeout=timeout_s)
                except FutureTimeoutError:
                    logger.warning("%s retriever timed out after %.1fs", name, timeout_s)
                    results[name] = []

        if not results["bm25"] and not results["vector"]:
            raise HybridSearchError(f"Both retrievers failed or timed out for query: {query_text!r}")
        return results["bm25"], results["vector"]


def preflight(hs: HybridSearch, probe: str = "How does session creation work?") -> None:
    """Fail loudly if the vector side is broken or points at a different index than BM25.
    A BM25-only run looks like a normal run, so run this before any eval."""
    if not hs.bm25_retriever.query(probe, k=5):
        raise RuntimeError("BM25 returned nothing")
    vector_hits = hs.vector_retriever.query(probe, k=5)
    if not vector_hits:
        raise RuntimeError("Vector retriever returned nothing (wrong collection name or model?)")
    bm25_ids = {m.get("chunk_id") for m in hs.bm25_retriever.metadata}
    unknown = {d["chunk_id"] for d in vector_hits} - bm25_ids
    if unknown:
        raise RuntimeError("Vector collection and BM25 index hold different chunks (stale collection); "
                           "set REINGEST = True and rebuild both.")
    print(f"preflight OK: {len(bm25_ids)} chunks in BM25, vector side consistent")


class Reranker:
    """Cross-encoder reranker. Adds `rerank_score` (raw logit) and returns the top_n, sorted."""

    def __init__(self, model_name: str = RERANK_MODEL, batch_size: int = 32, device: str | None = None):
        self.batch_size = batch_size
        self.model = CrossEncoder(model_name, device=device)

    def rerank(self, query: str, candidates: list[dict[str, Any]], top_n: int = 5,
               min_score: float | None = None, text_key: str = "text") -> list[dict[str, Any]]:
        if not candidates:
            return []
        pairs = [(query, c[text_key]) for c in candidates]
        try:
            scores = []
            for i in range(0, len(pairs), self.batch_size):
                scores.extend(float(s) for s in self.model.predict(pairs[i:i + self.batch_size]))
        except Exception:
            logger.exception("Reranking failed for query=%r; falling back to fused order", query)
            return [{**c, "rerank_score": c.get("fused_score", 0.0)} for c in candidates[:top_n]]

        scored = sorted(({**c, "rerank_score": s} for c, s in zip(candidates, scores)),
                        key=lambda c: c["rerank_score"], reverse=True)
        if min_score is not None:
            scored = [c for c in scored if c["rerank_score"] >= min_score]
        return scored[:top_n]


vector_retriever = VectorRetriever(collection, model)
hybrid_search = HybridSearch(bm25, vector_retriever)
re_ranker = Reranker()
preflight(hybrid_search)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1572.85it/s]


preflight OK: 184 chunks in BM25, vector side consistent


## 5. Adaptive RAG pipeline
Query complexity → retrieval budget → hybrid retrieval → rerank → adaptive cutoff → token budget → LLM.

In [21]:
COMPLEXITY_PROMPT = """
You are a query complexity classifier for a code repository.

Your task is to determine how much repository context is likely
required to answer the user's query.

Classify the query into exactly one of these levels:

LOW:
- Can probably be answered from one file, function, class, or
  small local section.
- Does not require significant cross-file reasoning.

MEDIUM:
- Requires understanding multiple related files, functions,
  or components.
- May require following a limited data or execution flow.

HIGH:
- Requires understanding multiple components or subsystems.
- Requires tracing a multi-step execution or data flow.
- Requires architectural or dependency reasoning.
- Requires understanding how several parts of the repository interact.

Consider:
1. Number of components involved
2. Number of files likely to be required
3. Whether the query requires tracing a flow
4. Whether cross-file reasoning is required
5. Whether architectural reasoning is required
6. Whether multiple steps need to be understood
7. Whether the query asks for comparison or impact analysis

Do not classify based only on query length.

Return ONLY valid JSON:

{{
    "complexity": "LOW | MEDIUM | HIGH",
    "confidence": 0.0,
    "reason": "Brief explanation"
}}

User query:
{query}
"""


def classify_complexity_heuristic(query: str) -> str | None:
    """Free heuristic. Returns None when unsure, which triggers the LLM classifier."""
    q = query.lower().strip()
    word_count = len(q.split())

    if any(s in q for s in ("trace", "end to end", "end-to-end", "architecture", "across", "interact")):
        return "HIGH"

    low_signals = ("what is", "where is", "define", "which file", "what does")
    if word_count <= 10 and (any(s in q for s in low_signals) or q.startswith(("what ", "where ", "which "))):
        return "LOW"

    if word_count > 25 or " compare " in q or "impact" in q:
        return "HIGH"
    if word_count <= 15:
        return None
    return "MEDIUM"


def classify_complexity(query: str, classifier) -> tuple[str, float, str]:
    heuristic = classify_complexity_heuristic(query)
    if heuristic is not None:
        return heuristic, 1.0, "heuristic"
    try:
        response = classifier.invoke(COMPLEXITY_PROMPT.format(query=query), max_tokens=256,
                                     response_format={"type": "json_object"})
        data = json.loads(response.content)
        complexity = str(data.get("complexity", "MEDIUM")).upper()
        if complexity not in RETRIEVAL_BUDGET:
            complexity = "MEDIUM"
        return complexity, float(data.get("confidence", 0.5)), data.get("reason", "llm_fallback")
    except Exception as e:
        print(f"[complexity] LLM unavailable; using MEDIUM: {e}")
        return "MEDIUM", 0.0, "llm_fallback_unavailable"


def adaptive_cutoff(reranked: list[dict], complexity: str = "MEDIUM", max_keep: int = 10) -> list[dict]:
    """Keep at least MIN_KEEP chunks, then stop at the first sharp relative drop in rerank_score.
    Relative drop uses |prev| as denominator because cross-encoder logits can be negative."""
    dropoff = DROPOFF_BY_COMPLEXITY.get(complexity, 0.12)
    min_keep = MIN_KEEP_BY_COMPLEXITY.get(complexity, 1)
    if len(reranked) <= min_keep:
        return reranked

    kept = [reranked[0]]
    for i in range(1, min(len(reranked), max_keep)):
        if len(kept) >= min_keep:
            prev, curr = reranked[i - 1]["rerank_score"], reranked[i]["rerank_score"]
            if (prev - curr) / max(abs(prev), 1e-6) > dropoff:
                break
        kept.append(reranked[i])
    return kept


enc = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    return len(enc.encode(text))


def trim_to_token_budget(docs: list[dict], reserved_output_tokens: int = 512,
                         prompt_overhead_tokens: int = 150, safety_margin: int = 300) -> list[dict]:
    budget = max(MODEL_CONTEXT_WINDOW - reserved_output_tokens - prompt_overhead_tokens - safety_margin, 200)
    kept, total = [], 0
    for d in docs:
        t = count_tokens(d["text"])
        if kept and total + t > budget:
            break
        kept.append(d)
        total += t
    return kept


def _generate(llm, query: str, context: str) -> str:
    prompt = f"""Answer the question using only the repository context below.

Answer directly in 1–3 concise sentences.
Use relevant file names, functions, classes, endpoints, and code from the context.
Do not use outside knowledge.

Repository context:
{context}

Question:
{query}

Answer:"""
    content = llm.invoke(prompt).content
    if isinstance(content, list):
        content = "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    answer = re.sub(r"<think>.*?</think>", "", str(content or ""), flags=re.DOTALL).strip()
    return answer or ABSTENTION


def ragPipeline(query, hybrid_search, reranker, llm, top_k=None, top_n=None,
                min_score=0.2, return_context=False, use_adaptive=True):
    """Hybrid retrieval + rerank + adaptive cutoff + LLM generation.
    top_k: candidates to retrieve (None → chosen by query complexity). top_n: kept after rerank."""
    complexity, complexity_conf, complexity_reason = classify_complexity(query, classifier_llm)
    fetch_k = top_k or RETRIEVAL_BUDGET.get(complexity, RETRIEVAL_BUDGET["MEDIUM"])

    def result(answer, sources=(), confidence=0.0, n_chunks=0, context=""):
        out = {"answer": answer, "sources": list(sources), "confidence": confidence,
               "complexity": complexity, "complexity_confidence": complexity_conf,
               "complexity_reason": complexity_reason, "retrieval_k": fetch_k,
               "final_chunk_count": n_chunks}
        if return_context:
            out["context"] = context
        return out

    # 1. retrieve, drop weak fused hits (fused score normalised to [0, 1])
    max_rrf = (BM25_WEIGHT + VECTOR_WEIGHT) / (RRF_K + 1)
    docs = [d for d in hybrid_search.hybrid_retrieval(query, k=fetch_k)
            if d.get("fused_score", 0.0) / max_rrf >= min_score]
    if not docs:
        return result(ABSTENTION)

    # 2. rerank, then adaptive cutoff
    docs = reranker.rerank(query, docs, top_n=top_n or fetch_k)
    if use_adaptive:
        docs = adaptive_cutoff(docs, complexity)

    # 3. drop exact duplicates, fit the token budget
    seen, unique = set(), []
    for d in docs:
        key = (d["metadata"].get("file_path", "unknown"), d["text"].strip())
        if key not in seen:
            seen.add(key)
            unique.append(d)
    docs = trim_to_token_budget(unique, prompt_overhead_tokens=count_tokens(query) + 80,
                                reserved_output_tokens=512, safety_margin=200)
    if not docs:
        return result(ABSTENTION)

    context = "\n\n".join(d["text"] for d in docs).strip()
    sources = [{"source": d["metadata"].get("file_path", "unknown"),
                "page": d["metadata"].get("page", "unknown"),
                "score": d.get("fused_score", 0.0) / max_rrf,
                "preview": d["text"][:120] + "..."} for d in docs]
    confidence = max(s["score"] for s in sources)

    # 4. abstain if even the best chunk is judged irrelevant, otherwise generate
    if max(d["rerank_score"] for d in docs) < RELEVANCE_THRESHOLD:
        return result(ABSTENTION, sources, confidence, len(docs), context)
    try:
        answer = _generate(llm, query, context)
    except Exception as exc:
        return result(f"Generation failed: {exc}", context=context)
    return result(answer, sources, confidence, len(docs), context)


# Smoke test
out = ragPipeline("Where is the FastAPI entry point defined?", hybrid_search=hybrid_search,
                  reranker=re_ranker, llm=llm, return_context=True)
print(out["answer"])
print(f"{out['complexity']} | k={out['retrieval_k']} | chunks used={out['final_chunk_count']} "
      f"| context tokens={count_tokens(out['context'])}")

The FastAPI entry point is defined in api_server.py where the FastAPI app is created with `app = FastAPI(...)` (top-level in api_server.py).
LOW | k=10 | chunks used=3 | context tokens=732


## 6. Optional: Redis result cache
Keyed on repo + commit + index version + query, so a new commit or a re-chunk never serves stale answers.

In [22]:
import redis

redis_client = redis.Redis(host=os.environ.get("REDIS_HOST", "localhost"),
                           port=int(os.environ.get("REDIS_PORT", 6379)),
                           password=os.environ.get("REDIS_PASSWORD"), decode_responses=True)
CACHE_TTL_SECONDS = 60 * 60 * 24
INDEX_VERSION = f"{EMBED_MODEL}:{MAX_CHUNK_CHARS}"


def _cache_key(repo_name: str, commit_sha: str, query: str) -> str:
    raw = f"{repo_name}:{commit_sha}:{INDEX_VERSION}:{query.strip().lower()}"
    return "ragcache:" + hashlib.sha256(raw.encode()).hexdigest()


def ragPipeline_cached(query, repo_name, commit_sha, hybrid_search, reranker, llm, use_cache=True, **kwargs):
    key = _cache_key(repo_name, commit_sha, query)
    if use_cache:
        try:
            cached = redis_client.get(key)
            if cached:
                return {**json.loads(cached), "_cache_hit": True}
        except redis.RedisError as e:
            print(f"[cache] read skipped, Redis unavailable: {e}")

    result = ragPipeline(query, hybrid_search, reranker, llm, **kwargs)
    result["_cache_hit"] = False
    if use_cache:
        try:
            redis_client.setex(key, CACHE_TTL_SECONDS, json.dumps(result))
        except redis.RedisError as e:
            print(f"[cache] write skipped, Redis unavailable: {e}")
    return result

## 7. Evaluation
- **Answer eval**: fixed retrieval (top 20 → rerank 10) vs. the adaptive pipeline. Reports token reduction, judged accuracy, abstention.
- **Retrieval eval**: rank of the chunk that contains the answer (`expected_symbols`), for vector-only, hybrid, and after reranking.

The LLM judge scores faithfulness to the retrieved context, not correctness, so read it together with the retrieval eval.

In [23]:
EVAL_SET = [
    # LOW: one file / small section
    {"query": "What language is this repository written in?", "expected_complexity": "LOW"},
    {"query": "Where is the FastAPI entry point defined?", "expected_complexity": "LOW",
     "expected_symbols": ["app = FastAPI("]},
    {"query": "Which embedding model does the EmbeddingManager use?", "expected_complexity": "LOW"},
    {"query": "Which cross-encoder model is used for reranking?", "expected_complexity": "LOW",
     "expected_symbols": ["cross-encoder/ms-marco-MiniLM-L-6-v2"]},
    {"query": "How long does a server-side session last before it expires?", "expected_complexity": "LOW",
     "expected_symbols": ["timedelta(hours=24)"]},
    {"query": "What are the default values of top_k, top_n and min_score in QueryRequest?", "expected_complexity": "LOW",
     "expected_symbols": ["class QueryRequest"]},
    {"query": "How many log entries does the in-memory log buffer keep?", "expected_complexity": "LOW",
     "expected_symbols": ["_MAX_LOG_ENTRIES"]},
    {"query": "Which API endpoints are admin only?", "expected_complexity": "LOW"},

    # MEDIUM: several functions / one subsystem
    {"query": "How does session creation and validation work in the FastAPI backend?", "expected_complexity": "MEDIUM"},
    {"query": "How does require_admin enforce role-based access control?", "expected_complexity": "MEDIUM",
     "expected_symbols": ["def require_admin"]},
    {"query": "How are BM25 and vector results combined in hybrid search, and what weights are used?", "expected_complexity": "MEDIUM",
     "expected_symbols": ["def _reciprocal_rank_fusion"]},
    {"query": "What happens when a non-admin Google account tries to log in via /auth/verify?", "expected_complexity": "MEDIUM",
     "expected_symbols": ["Only the admin can log in"]},
    {"query": "How does the API server bootstrap the vector store and BM25 index at startup?", "expected_complexity": "MEDIUM",
     "expected_symbols": ["Bootstrapping document database"]},
    {"query": "How does the Streamlit app verify a session after the Google login redirect?", "expected_complexity": "MEDIUM",
     "expected_symbols": ["st.session_state.user = resp.json()"]},
    {"query": "How does the Redis cache work and what happens if Redis is offline?", "expected_complexity": "MEDIUM",
     "expected_symbols": ["setex"]},

    # HIGH: cross-file / architectural
    {"query": "How does authentication and session management work across the frontend, API middleware, and database layers?", "expected_complexity": "HIGH"},
    {"query": "Trace a query from the Streamlit UI through the API to the final answer generation.", "expected_complexity": "HIGH"},
    {"query": "How does an uploaded PDF become searchable in both ChromaDB and the BM25 index?", "expected_complexity": "HIGH",
     "expected_symbols": ["state.bm25_retriever.add"]},
    {"query": "How does the /api/query endpoint interact with the cache, the RAG pipeline and the response trimming for non-admin users?", "expected_complexity": "HIGH"},
    {"query": "How do the BM25 and vector retrievers run in parallel, and what happens if one of them fails or times out?", "expected_complexity": "HIGH",
     "expected_symbols": ["def _run_retrievers_with_fallback"]},
    {"query": "What differs between what admin users and public users see across the Streamlit UI and the API?", "expected_complexity": "HIGH"},

    # Unanswerable: the correct behaviour is to abstain / reject the false premise
    {"query": "What database engine is used to store user passwords?", "expected_complexity": "LOW", "unanswerable": True},
    {"query": "How does the project handle payment processing?", "expected_complexity": "LOW", "unanswerable": True},
]

ABSTENTION_PHRASES = [
    "cannot be determined", "not defined in", "does not contain", "no information", "not covered",
    "cannot find", "not present in the context", "elsewhere in the codebase",
    "not included in the provided context", "not stored", "does not store", "no database",
    "not used for", "not implemented",
]


def is_abstention(answer: str) -> bool:
    a = answer.lower()
    return any(p in a for p in ABSTENTION_PHRASES)


def judge_answer(query: str, answer: str, context: str, judge) -> float:
    """LLM judge: 0 / 0.25 / 0.5 / 0.75 / 1 for how well the answer addresses the question given the context."""
    prompt = f"""Rate how well the ANSWER addresses the QUESTION using only the CONTEXT.
Return only one number: 0, 0.25, 0.5, 0.75, or 1.

QUESTION: {query}
CONTEXT: {context[:12000]}
ANSWER: {answer}

Score:"""
    try:
        content = judge.invoke(prompt).content
        if isinstance(content, list):
            content = "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
        match = re.search(r"\b(?:1(?:\.0+)?|0\.75|0\.5|0\.25|0)\b", str(content))
        if not match:
            print(f"[judge] invalid response: {content!r}")
            return 0.0
        return float(match.group())
    except Exception as e:
        print(f"[judge] scoring failed: {e}")
        return 0.0


def score_unanswerable(query: str, answer: str, judge) -> float:
    """1.0 if the answer abstains or rejects the false premise, 0.0 if it fabricates specifics."""
    if is_abstention(answer):
        return 1.0
    prompt = f"""The QUESTION below assumes something that does not exist in this codebase.
A correct response either says the information isn't available, OR correctly explains
why the premise is false without inventing specifics not grounded in real context.
An INCORRECT response confidently states a specific, fabricated answer.

QUESTION: {query}
ANSWER: {answer}

Is this response correct (did it avoid fabricating an answer)? Reply only: yes or no."""
    try:
        return 1.0 if "yes" in judge.invoke(prompt).content.lower() else 0.0
    except Exception as e:
        print(f"[abstention-judge] scoring failed, defaulting to 0.0: {e}")
        return 0.0


def run_eval(eval_set, hybrid_search, reranker, llm, judge, sleep_between=1.5, out_csv="eval_results.csv"):
    """Fixed retrieval (baseline) vs adaptive cutoff: token reduction, judged accuracy, abstention."""
    preflight(hybrid_search)
    rows = []
    for item in eval_set:
        query, unanswerable = item["query"], item.get("unanswerable", False)
        common = dict(hybrid_search=hybrid_search, reranker=reranker, llm=llm,
                      top_k=20, top_n=10, return_context=True)

        t0 = time.monotonic()
        base = ragPipeline(query, use_adaptive=False, **common)
        base_latency = time.monotonic() - t0
        time.sleep(sleep_between)
        t0 = time.monotonic()
        adapt = ragPipeline(query, **common)
        adapt_latency = time.monotonic() - t0

        base_tokens = count_tokens(base.get("context", ""))
        adapt_tokens = count_tokens(adapt.get("context", ""))

        if unanswerable:
            base_score = score_unanswerable(query, base["answer"], judge)
            adapt_score = score_unanswerable(query, adapt["answer"], judge)
        else:
            base_score = judge_answer(query, base["answer"], base.get("context", ""), judge)
            adapt_score = judge_answer(query, adapt["answer"], adapt.get("context", ""), judge)

        if base_tokens <= 0:
            print(f"Skipping metrics for failed baseline: {query[:60]}")
            continue

        rows.append({
            "query": query,
            "unanswerable": unanswerable,
            "expected_complexity": item.get("expected_complexity"),
            "predicted_complexity": adapt["complexity"],
            "baseline_tokens": base_tokens,
            "adaptive_tokens": adapt_tokens,
            "token_reduction_pct": round(100 * (1 - adapt_tokens / base_tokens), 1),
            "baseline_score": base_score,
            "adaptive_score": adapt_score,
            "accuracy_retained_pct": min(100.0, round(100 * adapt_score / base_score, 1)) if base_score > 0 else None,
            "baseline_latency_s": round(base_latency, 2),
            "adaptive_latency_s": round(adapt_latency, 2),
            "baseline_answer": base["answer"],
            "adaptive_answer": adapt["answer"],
        })
        print(f"✓ {query[:60]}... | tokens {base_tokens}->{adapt_tokens} | score {base_score:.2f}->{adapt_score:.2f}")
        time.sleep(sleep_between)

    if rows:
        with open(out_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

    answerable = [r for r in rows if not r["unanswerable"]]
    unanswerable_rows = [r for r in rows if r["unanswerable"]]
    valid = [r for r in answerable if r["accuracy_retained_pct"] is not None]
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0

    avg_reduction = mean([r["token_reduction_pct"] for r in answerable])
    avg_retained = mean([r["accuracy_retained_pct"] for r in valid])
    print("\n===== SUMMARY =====")
    print(f"Avg token reduction (answerable): {avg_reduction:.1f}%")
    print(f"Avg accuracy retained (answerable): {avg_retained:.1f}%   "
          f"[mean judge score {mean([r['baseline_score'] for r in answerable]):.3f} -> "
          f"{mean([r['adaptive_score'] for r in answerable]):.3f}]")
    if unanswerable_rows:
        print(f"Correct abstention rate (unanswerable): {mean([r['adaptive_score'] for r in unanswerable_rows]) * 100:.1f}%")
    print("Accept: fewer tokens, accuracy held." if avg_retained >= 95 else "Reject: raise MIN_KEEP_BY_COMPLEXITY and rerun.")
    return rows


def show_answers(rows, unanswerable=False):
    """Side-by-side baseline vs adaptive answers, for spot checks."""
    for r in rows:
        if r["unanswerable"] != unanswerable:
            continue
        print(f"\nQ: {r['query']}\n  baseline: {r['baseline_answer']}\n  adaptive: {r['adaptive_answer']}")
        print(f"  tokens {r['baseline_tokens']} -> {r['adaptive_tokens']} ({r['token_reduction_pct']}%) "
              f"| scores {r['baseline_score']:.2f} -> {r['adaptive_score']:.2f}")


def first_hit_rank(symbols, docs):
    """1-based rank of the first chunk containing any expected symbol, else None."""
    for i, d in enumerate(docs, 1):
        blob = d["text"] + " " + str(d["metadata"].get("symbol_name", ""))
        if any(s in blob for s in symbols):
            return i
    return None


def rank_metrics(ranks):
    n = len(ranks)
    hit = lambda k: sum(r is not None and r <= k for r in ranks) / n
    return {"hit@1": hit(1), "hit@3": hit(3), "hit@5": hit(5), "hit@10": hit(10),
            "MRR": sum(1 / r for r in ranks if r) / n}


def run_retrieval_eval(items, hybrid_search, vector_retriever, reranker):
    """Where does the answer-bearing chunk rank? Vector-only, hybrid (top 10 / top 20), and after reranking."""
    preflight(hybrid_search)
    cols = {"vector@10": [], "hybrid@10": [], "hybrid@20": [], "reranked": []}
    print(f"{'query':52}" + "".join(f"{c:>11}" for c in cols))
    for it in items:
        symbols = it.get("expected_symbols")
        if not symbols:
            continue
        q = it["query"]
        cands = hybrid_search.hybrid_retrieval(q, k=20)
        ranks = {
            "vector@10": first_hit_rank(symbols, vector_retriever.query(q, k=10)),
            "hybrid@10": first_hit_rank(symbols, cands[:10]),
            "hybrid@20": first_hit_rank(symbols, cands),
            "reranked": first_hit_rank(symbols, reranker.rerank(q, cands, top_n=10)),
        }
        for c, r in ranks.items():
            cols[c].append(r)
        print(f"{q[:50]:52}" + "".join(f"{str(r or '-'):>11}" for r in ranks.values()))

    print()
    summary = {c: rank_metrics(r) for c, r in cols.items()}
    for c, m in summary.items():
        print(f"{c:>10}: " + "  ".join(f"{k}={v:.2f}" for k, v in m.items()))
    return summary

In [24]:
retrieval_summary = run_retrieval_eval(EVAL_SET, hybrid_search, vector_retriever, re_ranker)

preflight OK: 184 chunks in BM25, vector side consistent
query                                                 vector@10  hybrid@10  hybrid@20   reranked
Where is the FastAPI entry point defined?                     3          3          3          3
Which cross-encoder model is used for reranking?              2          3          3          2
How long does a server-side session last before it            2          2          2          1
What are the default values of top_k, top_n and mi            1          1          1          1
How many log entries does the in-memory log buffer            2         10         10          2
How does require_admin enforce role-based access c            1          1          1          1
How are BM25 and vector results combined in hybrid           10          -         16          5
What happens when a non-admin Google account tries            1          1          1          1
How does the API server bootstrap the vector store            6       

In [25]:
results = run_eval(EVAL_SET, hybrid_search, re_ranker, llm, judge_llm)

preflight OK: 184 chunks in BM25, vector side consistent
✓ What language is this repository written in?... | tokens 1573->1573 | score 1.00->1.00
✓ Where is the FastAPI entry point defined?... | tokens 1536->732 | score 1.00->1.00
✓ Which embedding model does the EmbeddingManager use?... | tokens 1439->59 | score 1.00->1.00
✓ Which cross-encoder model is used for reranking?... | tokens 2085->353 | score 1.00->1.00
✓ How long does a server-side session last before it expires?... | tokens 859->309 | score 1.00->1.00
✓ What are the default values of top_k, top_n and min_score in... | tokens 2630->702 | score 1.00->1.00
✓ How many log entries does the in-memory log buffer keep?... | tokens 616->154 | score 1.00->1.00
✓ Which API endpoints are admin only?... | tokens 2224->681 | score 1.00->1.00
✓ How does session creation and validation work in the FastAPI... | tokens 1613->954 | score 1.00->1.00
✓ How does require_admin enforce role-based access control?... | tokens 1769->1769 | score 1.0

In [26]:
show_answers(results)
show_answers(results, unanswerable=True)


Q: What language is this repository written in?
  baseline: The repository is written in Python (files use Python syntax: FastAPI, argparse, asynccontextmanager, import statements like from langchain_community.document_loaders, and .py-style function definitions).
  adaptive: The repository is written in Python (files show Python code, use of FastAPI, argparse, asynccontextmanager, and Python packages).
  tokens 1573 -> 1573 (0.0%) | scores 1.00 -> 1.00

Q: Where is the FastAPI entry point defined?
  baseline: The FastAPI entry point is defined in api_server.py where the FastAPI app is created as `app = FastAPI(...)` and launched with `uvicorn api_server:app --host 0.0.0.0 --port 8000`.
  adaptive: The FastAPI entry point is defined in api_server.py where the FastAPI app is created with app = FastAPI(...).
  tokens 1536 -> 732 (52.3%) | scores 1.00 -> 1.00

Q: Which embedding model does the EmbeddingManager use?
  baseline: The EmbeddingManager is initialized with the BAAI model "BAAI